# Dataset Prefetch

- HF `dataset` load FineWeb-Edu 100BT
- consumes approx 750GB disk space to download and extract!!

Note:

- Karpathy converts dataset into compressed parquet shards
- I am using raw HF dataset and eating the 750GB for now

In [ ]:
import datasets

In [2]:
dataset_hf_path = "HuggingFaceFW/fineweb-edu"
dataset_hf_name = "sample-100BT"
dataset_hf_split = "train"

In [3]:
dataset_hf_names = datasets.get_dataset_config_names(dataset_hf_path)
print(f"Dataset configs: {dataset_hf_names[:5]}")
assert dataset_hf_name in dataset_hf_names

Dataset configs: ['default', 'sample-10BT', 'sample-100BT', 'sample-350BT', 'CC-MAIN-2025-05']


In [4]:
dataset_hf_splits = datasets.get_dataset_split_names(dataset_hf_path, dataset_hf_name)
print(f"Dataset splits: {dataset_hf_splits}")
assert dataset_hf_split in dataset_hf_splits

Dataset splits: ['train']


In [5]:
# NOTE: Takes ~750GB disk space!!!
dataset = datasets.load_dataset(
    dataset_hf_path,
    name=dataset_hf_name,
    split=dataset_hf_split,
    # streaming=True,
)
dataset = dataset.shuffle(seed=42)  # Match nanochat repackage_data_reference.py seed

In [ ]:
# Print some examples matching Karpathy's sharding scheme
row_group_size = 1024                 # Karpathy packs data in 1024-document row groups
characters_per_shard = 250_000_000    # He uses ~250M characters per shard

total_chars = 0
doc_in_group_idx = 0
group_idx = 0
shard_idx = 0

num_printed = 0

for i, example in enumerate(dataset):
    text = example["text"]
    num_chars = len(text)

    if group_idx == 0 and doc_in_group_idx in (0, row_group_size - 1):
        print(f"{i:6d}:{shard_idx:4d}:{group_idx:4d}:{doc_in_group_idx:4d}: {text[:10]!r} ... {text[-10:]!r} (len={num_chars})")
        num_printed += 1

    total_chars += num_chars
    doc_in_group_idx += 1

    if doc_in_group_idx % row_group_size == 0:
        doc_in_group_idx = 0
        group_idx += 1
        if total_chars >= characters_per_shard:
            total_chars = 0
            group_idx = 0
            shard_idx += 1
    
    if num_printed >= 10:
        break

# Validate against NanoChat

In [6]:
import pyarrow.parquet as pq

In [7]:
# Requires running nanochat speedrun.sh
# Especially nanochat script: python -m nanochat.dataset -n 240
for shard2 in range(5):
    pf = pq.ParquetFile(f"/home/user/.cache/nanochat/base_data/shard_{shard2:05d}.parquet")
    group_idx2 = 0
    rg = pf.read_row_group(group_idx2)
    batch = rg.column('text').to_pylist()
    text = batch[0]
    print(f"{shard2:4d}:{group_idx2:4d}: {text[:10]!r} ... {text[-10:]!r} (len={len(text)})")
    text = batch[-1]
    print(f"{shard2:4d}:{group_idx2:4d}: {text[:10]!r} ... {text[-10:]!r} (len={len(text)})")

   0:   0: 'Shipment &' ... 'al impact.' (len=8657)
   0:   0: '“Human pop' ... ' like one.' (len=3136)
   1:   0: 'How to run' ... ' reserved.' (len=978)
   1:   0: 'Textbooks ' ... 'ce system.' (len=9009)
   2:   0: 'Recycling ' ... 'ample.com.' (len=6257)
   2:   0: 'The Ancien' ... 'lic Online' (len=4182)
   3:   0: 'Chapter 1 ' ... 's - Part 2' (len=4736)
   3:   0: 'Research h' ... ' remedies.' (len=3795)
   4:   0: 'Franklin H' ... ' rarities.' (len=2241)
   4:   0: 'NEW YORK (' ... 'gagement.”' (len=5369)


# Save Parquet Files

- should be bit-for-bit identical to reference dataset:
- https://huggingface.co/datasets/karpathy/fineweb-edu-100b-shuffle

In [21]:
import os
import time
import json
import pyarrow as pa
assert pa.__version__ == '21.0.0'  # bitwise parity with Nanochat

In [25]:
output_dir = os.path.expanduser("~/.cache/marcin/fineweb-edu-100b-shuffle")
rowgroup_index_filepath = os.path.join(output_dir, 'rowgroup_index.json')
os.makedirs(output_dir, exist_ok=True)

In [26]:
chars_per_shard = 250_000_000
row_group_size = 1024

shard_idx = 0
shard_docs = []
shard_num_chars = 0
shard_hf_start_idx = 0
rowgroup_index = []
time_start = time.time()
for i, example in enumerate(dataset):
    text = example["text"]
    shard_docs.append(text)
    shard_num_chars += len(text)
    if shard_num_chars > chars_per_shard and len(shard_docs) % row_group_size == 0:
        shard_table = pa.Table.from_pydict({'text': shard_docs})
        shard_filename = f'shard_{shard_idx:05d}.parquet'
        shard_path = os.path.join(output_dir, shard_filename)
        pq.write_table(
            table=shard_table,
            where=shard_path,
            row_group_size=row_group_size,
            use_dictionary=False,
            compression="zstd",
            compression_level=3,
            write_statistics=False,
        )
        rowgroup_index.append({
            'filepath': shard_filename,
            'num_row_groups': len(shard_docs) // row_group_size,
            'start_idx': shard_hf_start_idx
        })
        with open(rowgroup_index_filepath, "w") as f:
            json.dump(rowgroup_index, f, indent=4)
        
        time_end = time.time()
        print(f'Written shard {shard_idx} time={time_end-time_start:.2f}s')
        time_start = time.time()
        shard_idx += 1
        shard_docs = []
        shard_num_chars = 0
        shard_hf_start_idx = i+1
        if shard_idx == 240:
            break

Written shard 0 time=7.84s
Written shard 1 time=8.35s
Written shard 2 time=35.92s
Written shard 3 time=15.88s
Written shard 4 time=57.78s
Written shard 5 time=62.02s
Written shard 6 time=48.08s
Written shard 7 time=40.11s
Written shard 8 time=44.88s
Written shard 9 time=48.33s
Written shard 10 time=46.87s
Written shard 11 time=46.51s
Written shard 12 time=49.77s
Written shard 13 time=48.31s
Written shard 14 time=47.83s
Written shard 15 time=49.12s
Written shard 16 time=47.89s
Written shard 17 time=47.64s
Written shard 18 time=49.69s
Written shard 19 time=49.51s
Written shard 20 time=49.91s
Written shard 21 time=48.89s
Written shard 22 time=48.45s
Written shard 23 time=49.36s
Written shard 24 time=48.26s
Written shard 25 time=50.49s
Written shard 26 time=48.54s
Written shard 27 time=48.32s
Written shard 28 time=48.64s
Written shard 29 time=49.93s
Written shard 30 time=48.55s
Written shard 31 time=53.11s
Written shard 32 time=51.31s
Written shard 33 time=48.08s
Written shard 34 time=48.0